# Parte de Luiz Henrique — Data Science

Projeto: homicídios de mulheres nas cinco grandes regiões do Brasil (2000–2023).

Este notebook contém as etapas atribuídas a Luiz na divisão proposta no notebook original: diagnóstico de qualidade, tratamento dos dados e plano de modelagem/avaliação.



## 0. Bibliotecas
Execute as células de cima para baixo, com a pasta `dados/` ao lado do notebook.
As bibliotecas estão em `requirements.txt`. O código usa os dados locais, sem baixar novas versões.


In [ ]:
from pathlib import Path
import json, hashlib, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

BASE = Path.cwd()
if not (BASE / 'dados' / 'brutos').exists() and (BASE / 'data-science' / 'dados').exists():
    BASE = BASE / 'data-science'
assert (BASE / 'dados' / 'brutos').exists(), 'Abra a pasta data-science antes de executar.'
DADOS = BASE / 'dados'
BRUTOS = DADOS / 'brutos'
ANO_INICIO, ANO_FIM = 2000, 2023
COR, COR2 = '#157F83', '#E08B42'
pd.set_option('display.max_columns', 20)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .2})


def grafico(nome, leitura):
    plt.tight_layout()
    plt.show()
    display(Markdown('**Leitura:** ' + leitura))

print('Integrantes: Bruno Henrique e Luiz Henrique')
print('Recorte:', ANO_INICIO, 'a', ANO_FIM)
print('Versões: pandas', pd.__version__, '| numpy', np.__version__)


In [ ]:
manifesto = json.loads((DADOS / 'manifesto_fontes.json').read_text(encoding='utf-8'))
for m in manifesto:
    assert hashlib.sha256((BRUTOS / m['arquivo']).read_bytes()).hexdigest() == m['sha256']

def ler(nome):
    return pd.DataFrame(json.loads((BRUTOS / nome).read_text(encoding='utf-8')))

cont = ler('homicidios_regioes.json')
taxas = ler('taxas_regioes.json')
nacional = ler('homicidios_brasil.json')
taxa_nacional = ler('taxas_brasil.json')
geo = json.loads((BRUTOS / 'regioes_referencia.geojson').read_text(encoding='utf-8'))
ref = pd.DataFrame([{'regiao_id': int(f['properties']['code']),
                     'regiao': f['properties']['name']} for f in geo['features']])
print('Arquivos brutos verificados; dados regionais e nacionais carregados.')


## 3. Diagnóstico de qualidade
**Responsável proposto: Luiz Henrique — confirmar após revisão.**

Verificamos completude, validade, consistência, unicidade/integridade e acurácia. As verificações usam os dados brutos. Se o teste encontra zero problemas, registramos zero; não criamos erros para demonstrar limpeza.

Também procuramos textos como “Ignorado” e auditamos os valores 0 e 999. Em uma contagem, esses números podem ser válidos: só poderiam ser tratados como ausentes com confirmação no dicionário da fonte.


In [ ]:
def chave(s):
    return s.astype('string').str.strip().str.lower().map(
        lambda v: unicodedata.normalize('NFKD', v).encode('ascii', 'ignore').decode() if pd.notna(v) else v)

IGNORADOS = {'', 'ignorado', 'ign/branco', 'nao informado', 'n/i', 'null', 'nan'}
quadros = []
for nome, t in [('contagem', cont), ('taxa', taxas)]:
    for c in t.columns:
        quadros.append((nome, c, int(t[c].isna().sum()),
                        int(chave(t[c]).isin(IGNORADOS).sum()),
                        int(t[c].eq(0).sum()), int(t[c].eq(999).sum())))
completude = pd.DataFrame(quadros, columns=['fonte', 'coluna', 'nulos', 'ausentes textuais', 'valor 0', 'valor 999'])
display(completude)
print('Zeros e 999 são auditados, mas só seriam recodificados com respaldo do dicionário da fonte.')

cobertura = []
for nome, t in [('contagem', cont), ('taxa', taxas)]:
    x = t.copy()
    x['ano'] = pd.to_datetime(x['periodo'], utc=True, errors='coerce').dt.year
    keys = set(zip(x['regiao_id'], x['ano']))
    faltantes = [(r, a) for r in range(1, 6) for a in range(ANO_INICIO, ANO_FIM+1) if (r, a) not in keys]
    cobertura.extend((nome, r, a) for r, a in faltantes)
faltantes = pd.DataFrame(cobertura, columns=['fonte', 'regiao_id', 'ano'])
display(faltantes)
print(f'No painel de 120 regiões-ano, há {len(faltantes)} taxas ausentes por cobertura temporal da fonte.')


In [ ]:
auditoria = []
exemplos = []
for nome, t, sid in [('contagem', cont, 40), ('taxa', taxas, 52)]:
    v = pd.to_numeric(t['valor'], errors='coerce')
    dt = pd.to_datetime(t['periodo'], errors='coerce', utc=True)
    regras = {
        'valor não numérico preenchido': t['valor'].notna() & v.isna(),
        'valor negativo': v.lt(0),
        'data impossível ou ilegível': t['periodo'].notna() & dt.isna(),
        'ano fora de 1980–2026': dt.notna() & ~dt.dt.year.between(1980, 2026),
        'série incorreta': t['serie_id'].ne(sid),
        'nível diferente de região': t['tipo_regiao'].ne(2),
        'chave sem referência territorial': ~t['regiao_id'].isin(ref['regiao_id']),
    }
    if sid == 40:
        regras['contagem fracionária'] = v.notna() & v.mod(1).ne(0)
    for regra, mask in regras.items():
        auditoria.append((nome, regra, int(mask.sum())))
        if mask.any(): exemplos.append((nome, regra, t.loc[mask].head(3).to_dict('records')))
    auditoria.extend([(nome, 'duplicatas exatas excedentes', int(t.duplicated().sum())),
                      (nome, 'IDs repetidos excedentes', int(t['id'].duplicated().sum())),
                      (nome, 'região-período repetido excedente', int(t.duplicated(['regiao_id','periodo']).sum()))])
auditoria = pd.DataFrame(auditoria, columns=['fonte','teste','ocorrências'])
display(auditoria)
print('Exemplos de violações:', exemplos or 'Nenhuma violação nas regras testadas.')

# Exemplos reais, inclusive quando as regras passam.
display(cont.loc[cont['valor'].isin([cont['valor'].min(), cont['valor'].max()])])
display(taxas.loc[taxas['valor'].isin([taxas['valor'].min(), taxas['valor'].max()])])
print('Nome territorial da fonte:', ref.loc[ref['regiao_id'].eq(5),'regiao'].iloc[0])
print('Padronização proposta: Centro Oeste → Centro-Oeste; mesma região, sem criar nova categoria.')

meta = [json.loads((BRUTOS / n).read_text(encoding='utf-8'))['data']
        for n in ['metadados_homicidios.json','metadados_taxas.json']]
n_rotulos_corrompidos = sum('�' in m['Titulo'] for m in meta)
print('Títulos de metadados contendo caractere de substituição:', n_rotulos_corrompidos)
print('Não há limite superior arbitrário para contagens ou taxas: extremos serão avaliados por IQR.')


In [ ]:
# Conferência de agregação: consistência entre as séries da própria fonte.
c = cont.assign(ano=pd.to_datetime(cont['periodo'], utc=True).dt.year)
n = nacional.assign(ano=pd.to_datetime(nacional['periodo'], utc=True).dt.year)
conf = c.groupby('ano')['valor'].sum().rename('soma_regiões').to_frame().join(
    n.set_index('ano')['valor'].rename('Brasil'), how='outer')
conf['diferença'] = conf['soma_regiões'] - conf['Brasil']
display(conf.loc[ANO_INICIO:ANO_FIM])
divergencias = int(conf.loc[ANO_INICIO:ANO_FIM, 'diferença'].ne(0).sum())
resumo = pd.DataFrame([
    ('Completude', f'{int(completude.nulos.sum())} nulos explícitos; {len(faltantes)} taxas não disponíveis no recorte', 'Manter lacunas de cobertura e expor denominadores'),
    ('Validade', f'{int(auditoria.loc[~auditoria.teste.str.contains("duplic|repet"), "ocorrências"].sum())} violações nas regras especificadas', 'Sem recodificação de valores válidos'),
    ('Consistência', f'{divergencias} divergências região × Brasil; 1 nome territorial a padronizar; {n_rotulos_corrompidos} títulos de metadados com perda de caracteres', 'Usar referência territorial e rótulos documentados; preservar originais'),
    ('Unicidade / integridade', f'{int(auditoria.loc[auditoria.teste.str.contains("duplic|repet"), "ocorrências"].sum())} duplicatas nos testes e 0 chaves territoriais órfãs', 'Junções validadas e interrupção em conflitos'),
    ('Acurácia', 'Não estimada: sem auditoria independente dos registros de óbito', 'Conferência interna não prova correspondência com a realidade'),
], columns=['dimensão','evidência','decisão'])
display(resumo)


**Sobre a acurácia:** conferir a soma das regiões com o Brasil verifica consistência interna. Isso não prova que todos os óbitos foram registrados corretamente. Sem uma auditoria independente dos registros originais, não é possível medir a acurácia real nem quantificar a subnotificação com estas tabelas.


## 4. Tratamento dos dados
**Responsável proposto: Luiz Henrique — confirmar após revisão.**

Ordem utilizada: padronizar textos e datas; tratar ausentes e inválidos; verificar duplicatas; decidir sobre imputação; sinalizar outliers.

As cinco taxas de 2023 não estão disponíveis na série consultada. Mantemos os campos vazios, sem usar média, mediana ou zero, para não inventar taxas regionais. Os extremos válidos são mantidos. A regra do IQR é apenas uma sinalização descritiva.


In [ ]:
# 4.1 Cópias e padronização, antes de tratar valores ausentes.
regioes = ref.copy()
regioes['regiao'] = regioes['regiao'].str.strip().replace({'Centro Oeste':'Centro-Oeste'})
textos_alterados = int(regioes['regiao'].ne(ref['regiao']).sum())
preparados, eventos = {}, []
for nome, origem in [('homicidios_mulheres', cont), ('taxa_por_100_mil_mulheres', taxas)]:
    t = origem.copy(deep=True)
    t['periodo'] = pd.to_datetime(t['periodo'].astype('string').str.strip(), errors='coerce', utc=True)
    # 4.2 Converter apenas ausências textuais e medidas realmente inválidas.
    disfarce = chave(t['valor']).isin(IGNORADOS)
    v = pd.to_numeric(t['valor'].mask(disfarce), errors='coerce')
    invalido = v.lt(0) | (v.mod(1).ne(0) & v.notna() if nome == 'homicidios_mulheres' else False)
    t['valor'] = v.mask(invalido)
    if t['periodo'].isna().any():
        raise ValueError('Data ausente ou inválida: exige revisão antes de construir o ano.')
    t['ano'] = t['periodo'].dt.year.astype(int)
    # 4.3 Duplicatas exatas podem ser removidas. Conflitos de chave exigem revisão.
    antes = len(t)
    t = t.drop_duplicates()
    assert not t.duplicated(['regiao_id','ano']).any(), 'Conflito região-ano: não escolher primeira linha arbitrariamente.'
    eventos.append((nome, int(disfarce.sum()), int(invalido.sum()), antes-len(t)))
    preparados[nome] = t.loc[t['ano'].between(ANO_INICIO, ANO_FIM), ['regiao_id','ano','valor']].rename(columns={'valor':nome})

grade = pd.MultiIndex.from_product([range(1,6), range(ANO_INICIO,ANO_FIM+1)], names=['regiao_id','ano']).to_frame(index=False)
painel_antes = grade.merge(preparados['homicidios_mulheres'], on=['regiao_id','ano'], how='left', validate='one_to_one').merge(
    preparados['taxa_por_100_mil_mulheres'], on=['regiao_id','ano'], how='left', validate='one_to_one')
df = painel_antes.merge(regioes, on='regiao_id', how='left', validate='many_to_one')
assert df['regiao'].notna().all()
df['data_referencia'] = pd.to_datetime(df['ano'].astype(str)+'-01-01')
NUM = ['homicidios_mulheres','taxa_por_100_mil_mulheres']
# 4.4 Não imputar medidas; sinalizar os ausentes de forma transparente.
for c in NUM:
    df[c+'_ausente'] = df[c].isna()
    df[c+'_imputado'] = False
# 4.5 IQR global é uma triagem descritiva; unidades região-ano não são independentes.
outliers = []
for c in NUM:
    s = df[c].dropna()
    q1, q3 = s.quantile([.25,.75]); iqr = q3-q1
    inferior, superior = q1-1.5*iqr, q3+1.5*iqr
    df[c+'_outlier_iqr'] = df[c].lt(inferior) | df[c].gt(superior)
    outliers.append((c, q1, q3, iqr, inferior, superior, int(df[c+'_outlier_iqr'].sum())))
df = df.sort_values(['regiao_id','ano']).reset_index(drop=True)
display(pd.DataFrame(outliers, columns=['variável','Q1','Q3','IQR','limite inferior','limite superior','n sinalizados']))
display(df.loc[df[[c+'_outlier_iqr' for c in NUM]].any(axis=1), ['regiao','ano']+NUM].head(10))
print('Extremos válidos foram mantidos. Regiões mais populosas podem ter maiores contagens.')


In [ ]:
decisoes = pd.DataFrame([
    ('Padronização de textos', 'Centro Oeste → Centro-Oeste na referência', 'Grafia canônica sem mudar a região', f'{textos_alterados} rótulo de referência; 24 linhas recebem o nome padronizado'),
    ('Datas', 'ISO da API → datetime UTC; derivar ano e marcador 01/01', 'O dia técnico 15/01 não é data de ocorrência', f'{len(cont)+len(taxas)} períodos brutos parseados; 120 marcadores anuais derivados'),
    ('Ausentes e inválidos', 'Auditar sentinelas e negativos; preservar zero válido', 'Não confundir código com medida sem documentação', f'{sum(e[1] for e in eventos)} sentinelas e {sum(e[2] for e in eventos)} inválidos convertidos'),
    ('Duplicatas', 'Remover somente exatas; bloquear conflitos da chave', 'Evitar perda arbitrária de informação', f'{sum(e[3] for e in eventos)} linhas removidas'),
    ('Recorte e junção', 'Grade de 5 regiões × 24 anos; junções validadas', 'Tornar lacunas de publicação explícitas', '120 linhas; 5 taxas ausentes em 2023'),
    ('Imputação', 'Não preencher medidas; criar flags', 'Evitar inventar taxa publicada; falta é estrutural', '0 valores imputados; 5 taxas continuam ausentes'),
    ('Outliers', 'Sinalizar pela regra 1,5 × IQR e manter', 'Valor extremo válido não é erro', str({c:int(df[c+'_outlier_iqr'].sum()) for c in NUM})),
    ('Escalonamento', 'Não aplicado nesta entrega', 'Sem treinamento; manter unidades interpretáveis', 'Contagens e taxas preservadas'),
    ('Brutos', 'Não sobrescrever; conferir SHA-256', 'Garantir rastreabilidade', '7 arquivos originais preservados'),
], columns=['tema','o que foi feito','por quê','impacto'])
display(decisoes)
antes_depois = pd.DataFrame({
    'painel antes do tratamento final':[len(painel_antes), int(painel_antes.duplicated(['regiao_id','ano']).sum()),int(painel_antes[NUM].isna().sum().sum()),int(painel_antes['taxa_por_100_mil_mulheres'].isna().sum()),0],
    'painel tratado':[len(df),int(df.duplicated(['regiao_id','ano']).sum()),int(df[NUM].isna().sum().sum()),int(df['taxa_por_100_mil_mulheres'].isna().sum()),int(df[[c+'_imputado' for c in NUM]].sum().sum())]
}, index=['linhas região-ano','chaves duplicadas excedentes','ausentes nas duas medidas','taxas ausentes','valores imputados'])
display(antes_depois)
print('A tabela compara painéis com a mesma granularidade; as 390 linhas brutas pertencem a dois indicadores e coberturas distintas.')
colunas_csv = ['ano','regiao'] + NUM
df[colunas_csv].to_csv(DADOS / 'base_tratada.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
print('Base tratada salva separadamente em dados/base_tratada.csv.')


In [ ]:
dicionario_tratado = pd.DataFrame([
    ('ano', 'Ano de referência', 'temporal anual', 'ano'),
    ('regiao', 'Grande região brasileira', 'qualitativa nominal', 'sem unidade'),
    ('homicidios_mulheres', 'Quantidade registrada de homicídios de mulheres', 'quantitativa discreta', 'óbitos'),
    ('taxa_por_100_mil_mulheres', 'Taxa publicada pela fonte', 'quantitativa contínua', 'óbitos por 100 mil mulheres'),
], columns=['coluna','significado','tipo estatístico','unidade'])
display(dicionario_tratado)
print('As colunas auxiliares de ausência, imputação e outlier usadas no notebook são indicadores booleanos.')
print('Elas ajudam a conferir o tratamento e não fazem parte das quatro colunas do CSV final.')


## 6. Plano de modelagem e avaliação
**Responsável proposto: Luiz Henrique — confirmar após revisão.**

| Item | Plano para a próxima entrega |
|---|---|
| Objetivo | Estudar a previsão da contagem regional de homicídios de mulheres no ano seguinte |
| Variável-alvo | Quantidade registrada no ano seguinte, por região |
| Entradas candidatas | Região e contagens dos anos anteriores |
| Técnicas | Referência que repete o valor do ano anterior e regressão regularizada simples |
| Métricas | MAE e RMSE, comparando também os resultados por região |
| Validação | Reservar os anos finais para teste; validar respeitando a ordem temporal, sem embaralhar anos |
| Cuidados | Transformações ajustadas somente no treino; não usar valores futuros; poucos anos e mudanças na qualidade dos registros podem limitar o resultado |

Nenhum modelo foi treinado e nenhuma métrica preditiva foi calculada nesta entrega. O objetivo é agregado, não uma previsão de risco individual. Se o projeto exigir classificação individual, será necessário discutir outro recorte e um alvo realmente observado com o professor.
